# Dataset Creation for Forecasting

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INVERTER_RAW_PATH = "../data/bronze/inverter_raw.csv"
POI_METER_RAW_PATH = "../data/bronze/poi_meter_raw.csv"
METEO_STATION_RAW_PATH = "../data/bronze/meteo_station_raw.csv"


In [2]:
def load_csv(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    if not lines:
        return pd.DataFrame()

    # 1. Get header and count target columns
    header = lines[0].strip().split(',')
    num_cols = len(header)
    
    # 2. Find where the JSON is (it's always the column before the last 4)
    # The last 4 are always: ingestion_timestamp, source, processing_status, error_message
    json_idx = header.index('raw_json')
    cols_after_json = num_cols - (json_idx + 1)

    data_rows = []
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        
        parts = line.split(',')
        
        # Leading columns (before JSON)
        leading = parts[:json_idx]
        
        # Trailing columns - take exactly the last cols_after_json columns
        trailing = parts[-cols_after_json:]

        # Everything in between is the JSON
        json_parts = parts[json_idx:-cols_after_json]
        json_blob = ",".join(json_parts).strip('"')

        # Combine and ensure we only have the exact number of columns as the header
        full_row = leading + [json_blob] + trailing
        data_rows.append(full_row[:num_cols])

    # Drop columns that are entirely empty or whitespace-only like 'error_message'
    df = pd.DataFrame(data_rows, columns=header)
    df = df.replace(r"^\s*$", pd.NA, regex=True)
    df = df.dropna(axis=1, how='all')

    return df

## Load data

In [3]:
inverter_raw = load_csv(INVERTER_RAW_PATH)
meteo_station_raw = load_csv(METEO_STATION_RAW_PATH)
poi_meter_raw = load_csv(POI_METER_RAW_PATH)

In [4]:
inverter = inverter_raw[['inverter_id', 'state', 'timestamp', 'inverter_temp_c', \
    'ac_power_kw', 'ac_freq_hz', 'dc_power_kw', 'dc_voltage_v', 'dc_current_a', \
    'active_failures', 'healthy_strings', 'failed_strings']].copy()
meteo_station = meteo_station_raw[['timestamp', 'amb_temp_c', 'module_temp_c', \
    'wind_speed_ms', 'wind_dir_deg', 'humidity_percent', 'poa_irradiance_wm2']].copy()
poi_meter = poi_meter_raw[['timestamp', 'export_active_power_kw', 'import_active_power_kw', \
    'reactive_power_kvar', 'grid_voltage_l1_v', 'grid_frequency_hz', 'power_factor', \
    'active_failures', 'connection_issues']].copy()

In [5]:
inverter["timestamp"] = pd.to_datetime(inverter["timestamp"], format='ISO8601')
inverter["state"] = inverter["state"].astype(int)
inverter["inverter_temp_c"] = inverter["inverter_temp_c"].astype(float)
inverter["ac_power_kw"] = inverter["ac_power_kw"].astype(float)
inverter["ac_freq_hz"] = inverter["ac_freq_hz"].astype(float)
inverter["dc_power_kw"] = inverter["dc_power_kw"].astype(float)
inverter["dc_voltage_v"] = inverter["dc_voltage_v"].astype(float)
inverter["dc_current_a"] = inverter["dc_current_a"].astype(float)
inverter["active_failures"] = inverter["active_failures"].astype(int)
inverter["healthy_strings"] = inverter["healthy_strings"].astype(int)
inverter["failed_strings"] = inverter["failed_strings"].astype(int)

inverter = inverter.sort_values("timestamp").reset_index(drop=True)

In [6]:
meteo_station["timestamp"] = pd.to_datetime(meteo_station["timestamp"], format='ISO8601')
meteo_station["amb_temp_c"] = meteo_station["amb_temp_c"].astype(float)
meteo_station["module_temp_c"] = meteo_station["module_temp_c"].astype(float)
meteo_station["wind_speed_ms"] = meteo_station["wind_speed_ms"].astype(float)
meteo_station["wind_dir_deg"] = meteo_station["wind_dir_deg"].astype(float)
meteo_station["humidity_percent"] = meteo_station["humidity_percent"].astype(float)
meteo_station["poa_irradiance_wm2"] = meteo_station["poa_irradiance_wm2"].astype(float)

meteo_station = meteo_station.sort_values("timestamp").reset_index(drop=True)

In [7]:
poi_meter["timestamp"] = pd.to_datetime(poi_meter["timestamp"], format='ISO8601')
poi_meter["export_active_power_kw"] = poi_meter["export_active_power_kw"].astype(float)
poi_meter["import_active_power_kw"] = poi_meter["import_active_power_kw"].astype(float)
poi_meter["reactive_power_kvar"] = poi_meter["reactive_power_kvar"].astype(float)
poi_meter["grid_voltage_l1_v"] = poi_meter["grid_voltage_l1_v"].astype(float)
poi_meter["grid_frequency_hz"] = poi_meter["grid_frequency_hz"].astype(float)
poi_meter["power_factor"] = poi_meter["power_factor"].astype(float)
poi_meter["active_failures"] = poi_meter["active_failures"].astype(int)
poi_meter["connection_issues"] = poi_meter["connection_issues"].astype(bool)

poi_meter = poi_meter.sort_values("timestamp").reset_index(drop=True)

## Dataset for Statistical Models (ARIMA, SARIMA, etc.)


Statistical models typically work with time series data in a simpler format. We will need only the time variable and the target variable. These columns will need some preprocessing like filling empty gaps in the data, but will be done before modelling.

In [8]:
# Keep each inverter separate (per-inverter time series)
statistical_dataset = inverter[['inverter_id', 'timestamp', 'ac_power_kw', 'active_failures']].copy()
statistical_dataset.head()

,inverter_id,timestamp,ac_power_kw,active_failures
0,B1,2026-02-01 19:41:11.671261+00:00,0.0,0
1,B0,2026-02-01 19:41:11.671261+00:00,0.0,0
2,A0,2026-02-01 19:41:11.671261+00:00,0.0,0
3,A2,2026-02-01 19:41:11.671261+00:00,0.0,0
4,B2,2026-02-01 19:41:11.671261+00:00,0.0,0


Analyze the percentage of malfunctioning time-serie steps

In [9]:
# Percentage of data where active_failures is > 0 per inverter
failure_analysis = statistical_dataset.groupby('inverter_id').agg(
    total_records=('active_failures', 'count'),
    records_with_failures=('active_failures', lambda x: (x > 0).sum())
)

failure_analysis['failure_percentage'] = (failure_analysis['records_with_failures'] / failure_analysis['total_records'] * 100).round(2)

print("Failure analysis per inverter:")
print(failure_analysis)
print(f"\nOverall failure percentage: {(statistical_dataset['active_failures'] > 0).sum() / len(statistical_dataset) * 100:.2f}%")

Failure analysis per inverter:
             total_records  records_with_failures  failure_percentage
inverter_id                                                          
A0                  127708                    457                0.36
A1                  127695                   1200                0.94
A2                  127979                    963                0.75
A3                  127689                    547                0.43
B0                  127769                   1531                1.20
B1                  127816                    339                0.27
B2                  127835                   2231                1.75
B3                  127731                    653                0.51
C0                  127768                   1755                1.37
C1                  127671                   1110                0.87
C2                  127608                   1321                1.04
C3                  127652                    809          

We will remove malfunctioning time steps

In [10]:
# Remove values where active_failures is > 0
statistical_dataset = statistical_dataset[statistical_dataset['active_failures'] == 0].copy()
statistical_dataset.drop(columns=['active_failures'], inplace=True)

print(f"Dataset shape after removing failures: {statistical_dataset.shape}")

Dataset shape after removing failures: (1773985, 3)


Check the gap distribution in the data

In [11]:
# Calculate time differences between consecutive records per inverter
statistical_dataset = statistical_dataset.sort_values(['inverter_id', 'timestamp']).reset_index(drop=True)

# Calculate time gaps in seconds (avoid beeing more precise than seconds)
statistical_dataset['time_diff'] = statistical_dataset.groupby('inverter_id')['timestamp'].diff().dt.total_seconds().round(0)

# Analyze gaps (excluding first record of each inverter which will be NaT)
gaps = statistical_dataset['time_diff'].dropna()

print("Time gap statistics:")
print(f"Mean gap: {gaps.mean()}")
print(f"Median gap: {gaps.median()}")
print(f"Most common gap: {gaps.mode()[0] if len(gaps.mode()) > 0 else 'N/A'}")
print(f"\nGap distribution (top 10):")
print(gaps.value_counts().head(10))

# Drop the temporary column
statistical_dataset.drop(columns=['time_diff'], inplace=True)

Time gap statistics:
Mean gap: 45.46299122138975
Median gap: 45.0
Most common gap: 40.0

Gap distribution (top 10):
time_diff
40.0    59687
45.0    59493
56.0    59463
46.0    59461
54.0    59448
33.0    59372
48.0    59309
31.0    59268
37.0    59253
36.0    59229
Name: count, dtype: int64


Resample to a constant frequency of 1 minute

In [12]:
# Resample to 1 minute for each inverter
resampled_datasets = []

for inverter_id in statistical_dataset['inverter_id'].unique():
    inverter_data = statistical_dataset[statistical_dataset['inverter_id'] == inverter_id].copy()
    
    # Set timestamp as index
    inverter_data = inverter_data.set_index('timestamp')
    
    # Resample only the ac_power_kw column to 1 minute frequency
    ac_power_resampled = inverter_data[['ac_power_kw']].resample('1min').mean()
    
    # Interpolate missing values (linear interpolation for AC power)
    ac_power_resampled['ac_power_kw'] = ac_power_resampled['ac_power_kw'].interpolate(method='linear', limit_direction='both')
    
    # Add inverter_id back as a column
    ac_power_resampled['inverter_id'] = inverter_id
    
    # Reset index to get timestamp back as a column
    ac_power_resampled = ac_power_resampled.reset_index()
    
    resampled_datasets.append(ac_power_resampled)
    
    print(f"Inverter {inverter_id}: resampled {len(ac_power_resampled)} records")

# Combine all inverters
statistical_dataset = pd.concat(resampled_datasets, ignore_index=True)

# Sort by inverter_id and timestamp
statistical_dataset = statistical_dataset.sort_values(['inverter_id', 'timestamp']).reset_index(drop=True)

print(f"\nAfter resampling:")
print(f"Total records: {len(statistical_dataset)}")
print(f"Missing values in ac_power_kw: {statistical_dataset['ac_power_kw'].isna().sum()}")
statistical_dataset.head(10)

Inverter A0: resampled 96010 records
Inverter A1: resampled 96012 records
Inverter A2: resampled 96014 records
Inverter A3: resampled 96012 records
Inverter B0: resampled 96014 records
Inverter B1: resampled 96013 records
Inverter B2: resampled 96013 records
Inverter B3: resampled 96013 records
Inverter C0: resampled 96013 records
Inverter C1: resampled 96013 records
Inverter C2: resampled 96012 records
Inverter C3: resampled 96011 records
Inverter C4: resampled 96012 records
Inverter C5: resampled 96013 records

After resampling:
Total records: 1344175
Missing values in ac_power_kw: 0


,timestamp,ac_power_kw,inverter_id
0,2026-02-01 19:41:00+00:00,0.0,A0
1,2026-02-01 19:42:00+00:00,0.0,A0
2,2026-02-01 19:43:00+00:00,0.0,A0
3,2026-02-01 19:44:00+00:00,0.0,A0
4,2026-02-01 19:45:00+00:00,0.0,A0
5,2026-02-01 19:46:00+00:00,0.0,A0
6,2026-02-01 19:47:00+00:00,0.0,A0
7,2026-02-01 19:48:00+00:00,0.0,A0
8,2026-02-01 19:49:00+00:00,0.0,A0
9,2026-02-01 19:50:00+00:00,0.0,A0


Check gaps after resampling

In [13]:
# Calculate time differences between consecutive records per inverter
statistical_dataset = statistical_dataset.sort_values(['inverter_id', 'timestamp']).reset_index(drop=True)

# Calculate time gaps in seconds (avoid beeing more precise than seconds)
statistical_dataset['time_diff'] = statistical_dataset.groupby('inverter_id')['timestamp'].diff().dt.total_seconds().round(0)

# Analyze gaps (excluding first record of each inverter which will be NaT)
gaps = statistical_dataset['time_diff'].dropna()

print("Time gap statistics:")
print(f"Mean gap: {gaps.mean()}")
print(f"Median gap: {gaps.median()}")
print(f"Most common gap: {gaps.mode()[0] if len(gaps.mode()) > 0 else 'N/A'}")
print(f"\nGap distribution (top 10):")
print(gaps.value_counts().head(10))

# Drop the temporary column
statistical_dataset.drop(columns=['time_diff'], inplace=True)

Time gap statistics:
Mean gap: 60.0
Median gap: 60.0
Most common gap: 60.0

Gap distribution (top 10):
time_diff
60.0    1344161
Name: count, dtype: int64


## Dataset for Machine Learning Models

ML models benefit from feature engineering including lagged values, rolling statistics, and time-based features. We'll keep per-inverter granularity to capture individual inverter behavior.

In [14]:
# Start with inverter data
ml_dataset = inverter.copy()

# Merge with meteorological data
ml_dataset = pd.merge_asof(ml_dataset, meteo_station, on='timestamp', direction='nearest', tolerance=pd.Timedelta('1min'))


print(f"ML dataset shape before feature engineering: {ml_dataset.shape}")
ml_dataset.head()

ML dataset shape before feature engineering: (1788288, 18)


,inverter_id,state,timestamp,inverter_temp_c,ac_power_kw,ac_freq_hz,dc_power_kw,dc_voltage_v,dc_current_a,active_failures,healthy_strings,failed_strings,amb_temp_c,module_temp_c,wind_speed_ms,wind_dir_deg,humidity_percent,poa_irradiance_wm2
0,B1,2,2026-02-01 19:41:11.671261+00:00,20.90,0.0,0.0,0.0,0.0,0.0,0,12,0,9.97,9.97,2.22,127.9,54.7,0.0
1,B0,2,2026-02-01 19:41:11.671261+00:00,20.03,0.0,0.0,0.0,0.0,0.0,0,12,0,9.97,9.97,2.22,127.9,54.7,0.0
2,A0,2,2026-02-01 19:41:11.671261+00:00,20.85,0.0,0.0,0.0,0.0,0.0,0,12,0,9.97,9.97,2.22,127.9,54.7,0.0
3,A2,2,2026-02-01 19:41:11.671261+00:00,20.37,0.0,0.0,0.0,0.0,0.0,0,12,0,9.97,9.97,2.22,127.9,54.7,0.0
4,B2,2,2026-02-01 19:41:11.671261+00:00,19.42,0.0,0.0,0.0,0.0,0.0,0,12,0,9.97,9.97,2.22,127.9,54.7,0.0


In [15]:
# Time-based features
ml_dataset['hour'] = ml_dataset['timestamp'].dt.hour
ml_dataset['day_of_week'] = ml_dataset['timestamp'].dt.dayofweek
ml_dataset['day_of_month'] = ml_dataset['timestamp'].dt.day
ml_dataset['month'] = ml_dataset['timestamp'].dt.month
ml_dataset['quarter'] = ml_dataset['timestamp'].dt.quarter
ml_dataset['is_weekend'] = ml_dataset['day_of_week'].isin([5, 6]).astype(int)

# Cyclic encoding for time features (to preserve cyclical nature)
ml_dataset['hour_sin'] = np.sin(2 * np.pi * ml_dataset['hour'] / 24)
ml_dataset['hour_cos'] = np.cos(2 * np.pi * ml_dataset['hour'] / 24)
ml_dataset['day_of_week_sin'] = np.sin(2 * np.pi * ml_dataset['day_of_week'] / 7)
ml_dataset['day_of_week_cos'] = np.cos(2 * np.pi * ml_dataset['day_of_week'] / 7)
ml_dataset['month_sin'] = np.sin(2 * np.pi * ml_dataset['month'] / 12)
ml_dataset['month_cos'] = np.cos(2 * np.pi * ml_dataset['month'] / 12)

print("Time-based features created")
ml_dataset.head()

Time-based features created


,inverter_id,state,timestamp,inverter_temp_c,ac_power_kw,ac_freq_hz,dc_power_kw,dc_voltage_v,dc_current_a,active_failures,...,day_of_month,month,quarter,is_weekend,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos
0,B1,2,2026-02-01 19:41:11.671261+00:00,20.90,0.0,0.0,0.0,0.0,0.0,0,...,1,2,1,1,-0.965926,0.258819,-0.781831,0.62349,0.866025,0.5
1,B0,2,2026-02-01 19:41:11.671261+00:00,20.03,0.0,0.0,0.0,0.0,0.0,0,...,1,2,1,1,-0.965926,0.258819,-0.781831,0.62349,0.866025,0.5
2,A0,2,2026-02-01 19:41:11.671261+00:00,20.85,0.0,0.0,0.0,0.0,0.0,0,...,1,2,1,1,-0.965926,0.258819,-0.781831,0.62349,0.866025,0.5
3,A2,2,2026-02-01 19:41:11.671261+00:00,20.37,0.0,0.0,0.0,0.0,0.0,0,...,1,2,1,1,-0.965926,0.258819,-0.781831,0.62349,0.866025,0.5
4,B2,2,2026-02-01 19:41:11.671261+00:00,19.42,0.0,0.0,0.0,0.0,0.0,0,...,1,2,1,1,-0.965926,0.258819,-0.781831,0.62349,0.866025,0.5


In [16]:
# Determine your actual sampling frequency
ml_dataset = ml_dataset.sort_values(['inverter_id', 'timestamp']).reset_index(drop=True)
time_diffs = ml_dataset.groupby('inverter_id')['timestamp'].diff().dt.total_seconds()
sampling_freq_seconds = time_diffs.mode()[0] if len(time_diffs.mode()) > 0 else time_diffs.median()
sampling_freq_minutes = sampling_freq_seconds / 60

In [17]:
# Intraday lags (within 24 hours - capture daily patterns)
intraday_lags = [
    int(5 / sampling_freq_minutes),      # 5 minutes
    int(15 / sampling_freq_minutes),     # 15 minutes
    int(30 / sampling_freq_minutes),     # 30 minutes
    int(60 / sampling_freq_minutes),     # 1 hour
    int(120 / sampling_freq_minutes),    # 2 hours
    int(360 / sampling_freq_minutes),    # 6 hours (half-day)
]

# Daily lags (1 day - capture daily cyclical patterns)
daily_lags = [
    int(1440 / sampling_freq_minutes),   # 1 day
    int(2880 / sampling_freq_minutes),   # 2 days
]

# Weekly lags (7 days - capture weekly patterns if any)
weekly_lags = [
    int(10080 / sampling_freq_minutes),  # 7 days
]

lag_periods = [lag for lag in intraday_lags + daily_lags + weekly_lags if lag > 0]
lag_periods = sorted(list(set(lag_periods)))  # Remove duplicates and sort

In [18]:
# Rolling window sizes
# Short windows (minutes/hours) for high-frequency patterns
short_windows = [
    int(5 / sampling_freq_minutes),      # 5 minutes
    int(15 / sampling_freq_minutes),     # 15 minutes
    int(60 / sampling_freq_minutes),     # 1 hour
]

# Medium windows (hours) for trend patterns
medium_windows = [
    int(360 / sampling_freq_minutes),    # 6 hours
    int(720 / sampling_freq_minutes),    # 12 hours
]

# Long windows (daily/weekly) for seasonal patterns
long_windows = [
    int(1440 / sampling_freq_minutes),   # 1 day
]

window_sizes = [w for w in short_windows + medium_windows + long_windows if w > 0]
window_sizes = sorted(list(set(window_sizes)))  # Remove duplicates and sort


In [ ]:
# Lagged features (per inverter)
# Sort by inverter_id and timestamp to ensure correct lag calculation
ml_dataset = ml_dataset.sort_values(['inverter_id', 'timestamp']).reset_index(drop=True)

# Create lagged features for the target variable
for lag in lag_periods:
    ml_dataset[f'ac_power_kw_lag_{lag}'] = ml_dataset.groupby('inverter_id')['ac_power_kw'].shift(lag)

# Lagged features for key weather variables
for lag in lag_periods:
    ml_dataset[f'poa_irradiance_wm2_lag_{lag}'] = ml_dataset.groupby('inverter_id')['poa_irradiance_wm2'].shift(lag)
    ml_dataset[f'amb_temp_c_lag_{lag}'] = ml_dataset.groupby('inverter_id')['amb_temp_c'].shift(lag)

print("Lagged features created")
ml_dataset.head(10)

Lagged features created


,inverter_id,state,timestamp,inverter_temp_c,ac_power_kw,ac_freq_hz,dc_power_kw,dc_voltage_v,dc_current_a,active_failures,...,poa_irradiance_wm2_lag_174,amb_temp_c_lag_174,poa_irradiance_wm2_lag_523,amb_temp_c_lag_523,poa_irradiance_wm2_lag_2095,amb_temp_c_lag_2095,poa_irradiance_wm2_lag_4190,amb_temp_c_lag_4190,poa_irradiance_wm2_lag_14665,amb_temp_c_lag_14665
0,A0,2,2026-02-01 19:41:11.671261+00:00,20.85,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0,2,2026-02-01 19:41:44.136330+00:00,19.28,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A0,2,2026-02-01 19:42:32.506975+00:00,19.59,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0,2,2026-02-01 19:43:27.839696+00:00,19.25,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0,2,2026-02-01 19:44:12.934204+00:00,19.80,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,A0,2,2026-02-01 19:45:01.168789+00:00,19.94,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,A0,2,2026-02-01 19:46:00.237302+00:00,19.39,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,A0,2,2026-02-01 19:46:48.035225+00:00,20.73,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,A0,2,2026-02-01 19:47:31.877184+00:00,19.71,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,A0,2,2026-02-01 19:48:17.666147+00:00,19.24,0.0,0.0,0.0,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Rolling statistics (per inverter)
for window in window_sizes:
    # Rolling mean for AC power
    ml_dataset[f'ac_power_kw_rolling_mean_{window}'] = ml_dataset.groupby('inverter_id')['ac_power_kw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )
    
    # Rolling std for AC power
    ml_dataset[f'ac_power_kw_rolling_std_{window}'] = ml_dataset.groupby('inverter_id')['ac_power_kw'].transform(
        lambda x: x.rolling(window=window, min_periods=1).std()
    )
    
    # Rolling mean for irradiance
    ml_dataset[f'poa_irradiance_wm2_rolling_mean_{window}'] = ml_dataset.groupby('inverter_id')['poa_irradiance_wm2'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

print("Rolling statistics created")
ml_dataset.head(10)

Rolling statistics created


,inverter_id,state,timestamp,inverter_temp_c,ac_power_kw,ac_freq_hz,dc_power_kw,dc_voltage_v,dc_current_a,active_failures,...,poa_irradiance_wm2_rolling_mean_87,ac_power_kw_rolling_mean_523,ac_power_kw_rolling_std_523,poa_irradiance_wm2_rolling_mean_523,ac_power_kw_rolling_mean_1047,ac_power_kw_rolling_std_1047,poa_irradiance_wm2_rolling_mean_1047,ac_power_kw_rolling_mean_2095,ac_power_kw_rolling_std_2095,poa_irradiance_wm2_rolling_mean_2095
0,A0,2,2026-02-01 19:41:11.671261+00:00,20.85,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0
1,A0,2,2026-02-01 19:41:44.136330+00:00,19.28,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,A0,2,2026-02-01 19:42:32.506975+00:00,19.59,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A0,2,2026-02-01 19:43:27.839696+00:00,19.25,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A0,2,2026-02-01 19:44:12.934204+00:00,19.80,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,A0,2,2026-02-01 19:45:01.168789+00:00,19.94,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,A0,2,2026-02-01 19:46:00.237302+00:00,19.39,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,A0,2,2026-02-01 19:46:48.035225+00:00,20.73,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,A0,2,2026-02-01 19:47:31.877184+00:00,19.71,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,A0,2,2026-02-01 19:48:17.666147+00:00,19.24,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
# Additional features
# Ratio of DC to AC power (efficiency indicator)
ml_dataset['dc_to_ac_ratio'] = ml_dataset['dc_power_kw'] / (ml_dataset['ac_power_kw'] + 1e-6)

# Power per healthy string
ml_dataset['power_per_healthy_string'] = ml_dataset['ac_power_kw'] / (ml_dataset['healthy_strings'] + 1)

# Temperature difference
ml_dataset['temp_diff'] = ml_dataset['module_temp_c'] - ml_dataset['amb_temp_c']

# Interaction features
ml_dataset['irradiance_temp_interaction'] = ml_dataset['poa_irradiance_wm2'] * ml_dataset['amb_temp_c']

print("Additional features created")
print(f"\nFinal ML dataset shape: {ml_dataset.shape}")
print(f"Number of features: {ml_dataset.shape[1] - 1}")  # -1 for timestamp
ml_dataset.head()

Additional features created

Final ML dataset shape: (1788288, 79)
Number of features: 78


,inverter_id,state,timestamp,inverter_temp_c,ac_power_kw,ac_freq_hz,dc_power_kw,dc_voltage_v,dc_current_a,active_failures,...,ac_power_kw_rolling_mean_1047,ac_power_kw_rolling_std_1047,poa_irradiance_wm2_rolling_mean_1047,ac_power_kw_rolling_mean_2095,ac_power_kw_rolling_std_2095,poa_irradiance_wm2_rolling_mean_2095,dc_to_ac_ratio,power_per_healthy_string,temp_diff,irradiance_temp_interaction
0,A0,2,2026-02-01 19:41:11.671261+00:00,20.85,0.0,0.0,0.0,0.0,0.0,0,...,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
1,A0,2,2026-02-01 19:41:44.136330+00:00,19.28,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,A0,2,2026-02-01 19:42:32.506975+00:00,19.59,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A0,2,2026-02-01 19:43:27.839696+00:00,19.25,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A0,2,2026-02-01 19:44:12.934204+00:00,19.80,0.0,0.0,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# Check for missing values
print("Missing values per column:")
print(ml_dataset.isnull().sum()[ml_dataset.isnull().sum() > 0])

Missing values per column:
ac_power_kw_lag_7                   98
ac_power_kw_lag_21                 294
ac_power_kw_lag_43                 602
ac_power_kw_lag_87                1218
ac_power_kw_lag_174               2436
ac_power_kw_lag_523               7322
ac_power_kw_lag_2095             29330
ac_power_kw_lag_4190             58660
ac_power_kw_lag_14665           205310
poa_irradiance_wm2_lag_7            98
amb_temp_c_lag_7                    98
poa_irradiance_wm2_lag_21          294
amb_temp_c_lag_21                  294
poa_irradiance_wm2_lag_43          602
amb_temp_c_lag_43                  602
poa_irradiance_wm2_lag_87         1218
amb_temp_c_lag_87                 1218
poa_irradiance_wm2_lag_174        2436
amb_temp_c_lag_174                2436
poa_irradiance_wm2_lag_523        7322
amb_temp_c_lag_523                7322
poa_irradiance_wm2_lag_2095      29330
amb_temp_c_lag_2095              29330
poa_irradiance_wm2_lag_4190      58660
amb_temp_c_lag_4190              5866

## Save data

In [23]:
inverter.to_csv("../data/datasets/forecasting/inverter.csv", index=False)
meteo_station.to_csv("../data/datasets/forecasting/meteo_station.csv", index=False)
poi_meter.to_csv("../data/datasets/forecasting/poi_meter.csv", index=False)

statistical_dataset.to_csv("../data/datasets/forecasting/statistical_models_dataset.csv", index=False)
ml_dataset.to_csv("../data/datasets/forecasting/ml_models_dataset.csv", index=False)